# 02 — Análisis de tablas exportadas

Esta notebook analiza los TSV exportados desde SQL Server con un enfoque limitado: preparar PostGIS de media tensión para Montecarlo. Como recordatorio, en estos archivos las columnas están separadas por tabuladores.

El objetivo es responder tres preguntas concretas:

- ¿Existen los TSV de MT?
- ¿`TMP_SHAPEFILE.Datos_Objeto` trae pistas suficientes para reconstruir geometría?
- ¿`Objetos_Red` cruza con `TMP_SHAPEFILE` mediante `idcad/IDCAD` y `coop/COOP` normalizados?


## 1. Preparación

Ubicamos la raíz del proyecto y la carpeta de exports generada por SQL Server.


In [1]:
# pathlib permite trabajar con rutas de forma clara y portable.
from pathlib import Path

# pandas será la herramienta principal para revisar tablas exportadas.
import pandas as pd

# Buscamos la raíz del proyecto subiendo hasta encontrar archivos propios del repo.
# No dependemos del nombre de la carpeta para que funcione aunque el repo se renombre.
RAIZ = Path.cwd()
while RAIZ.parent != RAIZ:
    if (RAIZ / 'docker-compose.yml').exists() and (RAIZ / 'scripts' / 'gis_sqlserver.sh').exists():
        break
    RAIZ = RAIZ.parent

# Definimos la carpeta donde la notebook 01 deja los TSV exportados.
EXPORTS = RAIZ / 'sqlserver' / 'exports'
print('Carpeta de exports:', EXPORTS)


Carpeta de exports: /home/jovyan/work/sqlserver/exports


## 2. Verificar archivos exportados

Primero confirmamos qué TSV existen. Si faltan archivos del piloto, las celdas siguientes degradan con mensajes claros en vez de fallar.


In [2]:
# Listamos los TSV exportados por SQL Server.
# Esta verificación permite distinguir entre datos ausentes y errores de análisis.
archivos = sorted(EXPORTS.glob('*.tsv'))
print('Cantidad de TSV exportados:', len(archivos))
for path in archivos:
    print('-', path.name)


Cantidad de TSV exportados: 5
- mt_cables.tsv
- objetos_red.tsv
- seccionadores.tsv
- set.tsv
- tmp_shapefile.tsv


## 3. Cargar tablas disponibles

Cargamos cada TSV con `dtype=str` para no perder ceros a la izquierda ni convertir claves CAD a números. El tabulador se mantiene como separador porque los textos pueden contener comas.


In [3]:
# Cargamos todas las tablas disponibles en un diccionario: nombre -> DataFrame.
# Si aparece un TSV con problemas, se informa y se continúa con el resto.
tablas = {}
for path in archivos:
    nombre = path.stem.lower()
    try:
        tablas[nombre] = pd.read_csv(path, sep='	', dtype=str, keep_default_na=False)
        print(f'{nombre}: {len(tablas[nombre])} filas, {len(tablas[nombre].columns)} columnas')
    except Exception as exc:
        print(f'No se pudo leer {path.name}: {exc}')


mt_cables: 40 filas, 3 columnas
objetos_red: 69134 filas, 18 columnas
seccionadores: 1063 filas, 21 columnas
set: 800 filas, 17 columnas
tmp_shapefile: 125688 filas, 6 columnas


## 4. Enfoque de clasificación

La clasificación se limita a las tablas. `MT_cables.tsv` se interpreta como tabla de catálogo/dominio o atributos de soporte; no se asume que contenga la traza geométrica por sí sola.

La geometría inicial se busca en `TMP_SHAPEFILE.Datos_Objeto` y en el DXF `Montecarlo_MT.dxf`, no en `Trafos`, BT, suministros, alumbrado ni reclamos.


In [4]:
# Definimos el rol esperado de cada tabla del piloto MT.
# Esta matriz evita mezclar tablas de fases posteriores con la primera geometría PostGIS.
roles_piloto = pd.DataFrame([
    {'tabla': 'tmp_shapefile', 'rol': 'fuente geométrica cruda', 'uso': 'reconstruir POINT, LINESTRING o POLYGON desde Datos_Objeto'},
    {'tabla': 'objetos_red', 'rol': 'atributos de red', 'uso': 'enriquecer entidades usando idcad/coop normalizados'},
    {'tabla': 'set', 'rol': 'atributos MT', 'uso': 'validar y enriquecer entidades SET'},
    {'tabla': 'seccionadores', 'rol': 'atributos MT', 'uso': 'validar y enriquecer seccionadores'},
    {'tabla': 'mt_cables', 'rol': 'catálogo o soporte MT', 'uso': 'aportar atributos; no asumir geometría de traza'},
])
roles_piloto['exportado'] = roles_piloto['tabla'].isin(tablas.keys())
roles_piloto


,tabla,rol,uso,exportado
0,tmp_shapefile,fuente geométrica cruda,"reconstruir POINT, LINESTRING o POLYGON desde ...",True
1,objetos_red,atributos de red,enriquecer entidades usando idcad/coop normali...,True
2,set,atributos MT,validar y enriquecer entidades SET,True
3,seccionadores,atributos MT,validar y enriquecer seccionadores,True
4,mt_cables,catálogo o soporte MT,aportar atributos; no asumir geometría de traza,True


## 5. Resumen de tablas

Este resumen muestra filas, columnas y presencia de campos críticos. Si una tabla todavía no fue exportada, queda marcada como pendiente.


In [5]:
# Armamos un resumen limitado a las tablas del piloto MT.
# Revisamos columnas críticas sin asumir mayúsculas o minúsculas exactas.
tablas_objetivo = ['tmp_shapefile', 'objetos_red', 'set', 'seccionadores', 'mt_cables']
filas_resumen = []
for nombre in tablas_objetivo:
    df = tablas.get(nombre)
    if df is None:
        filas_resumen.append({'tabla': nombre, 'estado': 'pendiente', 'filas': 0, 'columnas': 0, 'campos_clave': ''})
        continue
    columnas = list(df.columns)
    campos_clave = [c for c in columnas if c.lower() in {'idcad', 'coop', 'datos_objeto'}]
    filas_resumen.append({'tabla': nombre, 'estado': 'exportada', 'filas': len(df), 'columnas': len(columnas), 'campos_clave': ', '.join(campos_clave)})

resumen_piloto = pd.DataFrame(filas_resumen)
resumen_piloto


,tabla,estado,filas,columnas,campos_clave
0,tmp_shapefile,exportada,125688,6,"COOP, IDCAD, Datos_Objeto"
1,objetos_red,exportada,69134,18,"idcad, coop"
2,set,exportada,800,17,"idcad, coop"
3,seccionadores,exportada,1063,21,"coop, idcad"
4,mt_cables,exportada,40,3,


## 6. Inspección de `TMP_SHAPEFILE.Datos_Objeto`

`Datos_Objeto` contiene texto de estilo DXF con tokens `§`. La inspección inicial busca entidades CAD que orientan la reconstrucción geométrica:

- `INSERT` y `CIRCLE` → punto.
- `LINE` y `LWPOLYLINE` abierta → línea.
- `LWPOLYLINE` cerrada → polígono.


In [6]:
# Buscamos una columna por nombre ignorando mayúsculas y minúsculas.
def buscar_columna(df: pd.DataFrame, candidatos: list[str]) -> str | None:
    # Creamos un mapa normalizado para encontrar columnas aunque cambie la capitalización.
    mapa = {col.lower(): col for col in df.columns}
    for candidato in candidatos:
        if candidato.lower() in mapa:
            return mapa[candidato.lower()]
    return None

# Clasificamos una cadena Datos_Objeto con reglas simples y explícitas para la clase.
def clasificar_datos_objeto(valor: str) -> str:
    # Normalizamos a mayúsculas para detectar tokens CAD sin depender del caso original.
    texto = str(valor).upper()
    if 'LWPOLYLINE' in texto:
        # En DXF el grupo 70 con valor 1 suele indicar polilínea cerrada.
        if '§70§1' in texto or '70§1' in texto or 'CLOSED' in texto:
            return 'LWPOLYLINE cerrada -> POLYGON'
        return 'LWPOLYLINE abierta -> LINESTRING'
    if 'INSERT' in texto:
        return 'INSERT -> POINT'
    if 'CIRCLE' in texto:
        return 'CIRCLE -> POINT'
    if 'LINE' in texto:
        return 'LINE -> LINESTRING'
    return 'sin entidad reconocida'

# Analizamos TMP_SHAPEFILE solo si fue exportada y contiene Datos_Objeto.
tmp = tablas.get('tmp_shapefile')
if tmp is None:
    print('TMP_SHAPEFILE no fue exportada todavía; ejecutar la notebook 01 con export-table.')
else:
    col_datos = buscar_columna(tmp, ['Datos_Objeto'])
    if col_datos is None:
        print('TMP_SHAPEFILE existe, pero no se encontró la columna Datos_Objeto.')
    else:
        tmp_analisis = tmp[[col_datos]].copy()
        tmp_analisis['tipo_reconstruccion'] = tmp_analisis[col_datos].map(clasificar_datos_objeto)
        print('Distribución de pistas de reconstrucción:')
        print(tmp_analisis['tipo_reconstruccion'].value_counts(dropna=False))
        tmp_analisis.head(10)


Distribución de pistas de reconstrucción:
tipo_reconstruccion
sin entidad reconocida              39567
INSERT -> POINT                     35759
LINE -> LINESTRING                  30452
LWPOLYLINE abierta -> LINESTRING    14949
CIRCLE -> POINT                      4904
LWPOLYLINE cerrada -> POLYGON          57
Name: count, dtype: int64


## 7. Cobertura de unión entre `Objetos_Red` y `TMP_SHAPEFILE`

La unión esperada se calcula con claves normalizadas:

- `Objetos_Red.idcad` con `TMP_SHAPEFILE.IDCAD`.
- `Objetos_Red.coop` con `TMP_SHAPEFILE.COOP`.

La exploración previa indicó una cobertura aproximada de 68,66%. Esta celda permite confirmar el valor con los TSV disponibles.


In [7]:
# Normalizamos claves CAD para comparar valores aunque vengan con espacios, mayúsculas o sufijos numéricos simples.
def normalizar_clave(serie: pd.Series) -> pd.Series:
    # Convertimos a texto, quitamos espacios, pasamos a mayúsculas y removemos un sufijo .0 frecuente en exports numéricos.
    return serie.astype(str).str.strip().str.upper().str.replace(r'\.0$', '', regex=True)

# Calculamos cobertura de unión solo si ambas tablas y columnas están disponibles.
objetos = tablas.get('objetos_red')
tmp = tablas.get('tmp_shapefile')
if objetos is None or tmp is None:
    print('Falta objetos_red.tsv o tmp_shapefile.tsv; no se puede medir cobertura de unión todavía.')
else:
    col_obj_idcad = buscar_columna(objetos, ['idcad', 'IDCAD'])
    col_obj_coop = buscar_columna(objetos, ['coop', 'COOP'])
    col_tmp_idcad = buscar_columna(tmp, ['IDCAD', 'idcad'])
    col_tmp_coop = buscar_columna(tmp, ['COOP', 'coop'])
    columnas_faltantes = [nombre for nombre, col in {
        'Objetos_Red.idcad': col_obj_idcad,
        'Objetos_Red.coop': col_obj_coop,
        'TMP_SHAPEFILE.IDCAD': col_tmp_idcad,
        'TMP_SHAPEFILE.COOP': col_tmp_coop,
    }.items() if col is None]
    if columnas_faltantes:
        print('Faltan columnas para medir cobertura:', ', '.join(columnas_faltantes))
    else:
        objetos_claves = pd.DataFrame({
            'idcad_norm': normalizar_clave(objetos[col_obj_idcad]),
            'coop_norm': normalizar_clave(objetos[col_obj_coop]),
        })
        tmp_claves = pd.DataFrame({
            'idcad_norm': normalizar_clave(tmp[col_tmp_idcad]),
            'coop_norm': normalizar_clave(tmp[col_tmp_coop]),
        }).drop_duplicates()
        objetos_validos = objetos_claves[(objetos_claves['idcad_norm'] != '') & (objetos_claves['coop_norm'] != '')]
        unidos = objetos_validos.merge(tmp_claves, on=['idcad_norm', 'coop_norm'], how='left', indicator=True)
        cobertura = (unidos['_merge'].eq('both').mean() * 100) if len(unidos) else 0
        print(f'Registros de Objetos_Red con claves válidas: {len(objetos_validos)}')
        print(f'Cobertura de unión contra TMP_SHAPEFILE: {cobertura:.2f}%')
        print('Referencia esperada de exploración previa: aproximadamente 68,66%.')


Registros de Objetos_Red con claves válidas: 69134
Cobertura de unión contra TMP_SHAPEFILE: 68.65%
Referencia esperada de exploración previa: aproximadamente 68,66%.


## 8. Lectura de `MT_cables`

`MT_cables.tsv` puede aportar atributos de catálogo, codificación o soporte operativo. En esta etapa no se lo toma como geometría de traza hasta confirmar una clave o vínculo claro con CAD/DXF.


In [8]:
# Revisamos MT_cables de forma conservadora: columnas, filas y posibles claves.
# No inferimos geometría de traza desde esta tabla sin evidencia adicional.
mt_cables = tablas.get('mt_cables')
if mt_cables is None:
    print('mt_cables.tsv no fue exportado todavía.')
else:
    columnas_interes = [c for c in mt_cables.columns if c.lower() in {'id', 'idcad', 'coop', 'codigo', 'tipo', 'descripcion'}]
    print('Filas en mt_cables:', len(mt_cables))
    print('Columnas de posible interés:', columnas_interes)
    mt_cables.head(10)


Filas en mt_cables: 40
Columnas de posible interés: ['Tipo']


## 9. Cierre del análisis de tablas

En esta notebook se revisaron los TSV exportados, se confirmó el alcance del piloto MT y se evaluaron pistas de reconstrucción en `TMP_SHAPEFILE.Datos_Objeto` junto con la cobertura de unión contra `Objetos_Red`.

- Las tablas del piloto quedaron identificadas y separadas de dominios posteriores.
- La notebook 03 contrasta estas pistas con `dxf/Montecarlo_MT.dxf` y sus capas MT.
